In [2]:
# 라이브러리 불러오기
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import os
from google.colab import drive

# Google Drive 연동 및 작업 디렉토리 설정
drive.mount('/content/drive')
os.chdir("/content/drive/MyDrive/비어플/dataset")

# 데이터 불러오기
df_pop = pd.read_csv("202502_202502_주민등록인구및세대현황_월간.csv", encoding="cp949")
df_vac = pd.read_excel("빈집실태조사 결과 현황 조회.xlsx")
df_farm = pd.read_excel("시도별_·_이동유형별_귀농가구원_20250327003915.xlsx")
df_decline = pd.read_excel("인구감소지역비율.xlsx")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# 필요한 컬럼 선택 및 이름 변경
df_pop = df_pop[['id', '2025년02월_총인구수']]
df_vac = df_vac[['id', '1등급(양호)', '2등급(일반)', '3등급(불량)', '주택 총 수']]
df_farm = df_farm[['id', '귀농인 수']]
df_decline = df_decline[['id', '시군구 수', '인구감소지역 수']]

df = df_pop.merge(df_vac, on='id', how='outer') \
           .merge(df_farm, on='id', how='outer') \
           .merge(df_decline, on='id', how='outer')

# 전국 제거 및 컬럼명 영어로
df = df[df['id'] != '전국'].copy()
df.columns = ['id', 'total_population', 'grade1', 'grade2', 'grade3', 'total_houses',
              'returning_farmers', 'total_regions', 'decline_regions']

# 문자열 숫자 → 숫자형으로 변환
for col in df.columns[1:]:
    if df[col].dtype == 'object':
        df[col] = df[col].str.replace(',', '', regex=False)
    try:
        df[col] = df[col].astype(float)
        if (df[col] % 1 == 0).all():
            df[col] = df[col].astype(int)
    except:
        pass

# 파생 변수: 사용 가능한 빈집 수
df['grade_1or2'] = df['grade1'] + df['grade2']

# 점수 계산
score = pd.DataFrame({'id': df['id']})
score['score1'] = df['returning_farmers'] / df['total_population'] * 100000
score['score2'] = df['grade_1or2'] / df['total_houses'] * 100000
score['score3'] = df['grade_1or2'] / df['total_population'] * 100000
score['score4'] = df['decline_regions'] / df['total_regions']

# 결측치 처리
score['score1'] = score['score1'].fillna(0)
score = score.fillna(score.mean(numeric_only=True))

# 정규화 및 종합 점수
cols = ['score1', 'score2', 'score3', 'score4']
scaled_cols = ['scaled_' + c for c in cols]
scaler = MinMaxScaler()
score[scaled_cols] = scaler.fit_transform(score[cols])

score['total_score'] = score[scaled_cols].mean(axis=1)
score.sort_values('total_score', ascending=False, inplace=True)

# 저장
score.to_csv('지역선정기준 점수.csv', index=False, encoding='utf-8-sig')

# 출력 확인
score


,id,score1,score2,score3,score4,scaled_score1,scaled_score2,scaled_score3,scaled_score4,total_score
13,전남,125.455968,380.188456,175.470334,0.727273,1.000000,0.848632,0.895397,1.000000,0.936007
3,경북,97.105188,444.600455,195.200841,0.681818,0.774018,1.000000,1.000000,0.937500,0.927880
14,전북,79.596175,388.147983,168.990575,0.714286,0.634455,0.867337,0.861045,0.982143,0.836245
2,경남,48.274954,383.011515,156.001058,0.611111,0.384796,0.855266,0.792180,0.840278,0.718130
17,충북,59.807296,301.838485,125.148811,0.545455,0.476719,0.664510,0.628615,0.750000,0.629961
15,제주,52.678444,378.861281,143.369176,0.355464,0.419896,0.845513,0.725211,0.488763,0.619846
16,충남,82.141486,155.206803,65.048189,0.600000,0.654744,0.319927,0.309987,0.825000,0.527414
7,부산,1.410032,412.216214,165.310941,0.187500,0.011239,0.923897,0.841537,0.257812,0.508621
10,울산,9.217195,436.330922,159.338844,0.000000,0.073470,0.980567,0.809875,0.000000,0.465978
9,세종,15.829693,251.304833,101.138290,0.355464,0.126177,0.545757,0.501321,0.488763,0.415505
